# Explainable AI: understanding resale-price predictions

## Introduction

A price model can predict well while remaining difficult to understand. This notebook revisits *Explainable Artificial Intelligence* through three questions: which features does a fitted model rely on, how do its predictions vary with a feature, and why did it forecast this particular laptop's price?

The worked examples fit a tree ensemble, build a simple surrogate, explore permutation importance and PDP/ICE plots, and reconstruct an individual prediction from SHAP values. The application returns to the local laptop-resale data so you can focus on the explanations.

**Outline:** [Key Concepts](#key-concepts) · [XAI foundations](#xai-foundations) · [Global explanation methods](#global-explanation-methods) · [Local and example-based explanations](#local-and-example-based-explanations) · [Summary](#summary) · [Tasks](#tasks)

**Lecture connection:** slides 7–9 (stakeholders and interpretable models), 11–22 (surrogates, permutation importance and PDP), and 25–44 (local, SHAP and example-based explanations). Page references count PDF pages; some late-slide printed numbers differ.

We'll use **pandas** to organize the resale data, **NumPy** to reconstruct predictions, and **Matplotlib** to visualize the explanations. **Scikit-learn** supplies the model and global explanation tools, while **SHAP** provides the local contributions. The saved notebook includes the resulting plots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.metrics import mean_absolute_error
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

## Key Concepts

Let $\widehat{f}(\mathbf{x}_i)$ be a fitted resale-price forecast for observation $i$ with $d$ features. **Global** explanations summarize behavior across many observations; **local** explanations focus on a particular prediction.

*Permutation importance* measures the decrease in a chosen performance score when a feature is shuffled, holding the fitted model fixed. It is not the performance change from retraining without that feature.

For a feature $j$ and a candidate value $v$, define $\mathbf{x}_i^{(j\leftarrow v)}$ as row $i$ with feature $j$ replaced by $v$. A *partial dependence* average over $n$ reference rows is

$$\operatorname{PD}_j(v)=\frac{1}{n}\sum_{i=1}^{n}\widehat{f}(\mathbf{x}_i^{(j\leftarrow v)}).$$

An *ICE* curve keeps the individual predictions before averaging. It can reveal different model responses for different rows that a single average conceals.

For a local SHAP explanation, let $\phi_{i0}$ be the baseline output and $\phi_{ij}$ the contribution of feature $j$. The additive accounting is

$$\widehat{f}(\mathbf{x}_i)=\phi_{i0}+\sum_{j=1}^{d}\phi_{ij}.$$

Here all terms are in **USD**. Contributions depend on the model, the explained row and the background/missing-feature convention. They explain a model output relative to a reference; they do not establish causal effects or explain the full difference between prediction and observed price.

## XAI foundations

### Define the prediction time and the explanation audience

We forecast a laptop's resale price **at return, before sale**, using its original retail price (USD), actual lease duration (months) and measured battery capacity (%). These inputs are available at return, but not all would be known at contract signing.

A remarketing analyst might want a global view of price drivers. A colleague reviewing one forecast might want a local explanation. Both need understandable units and a clear account of what the model actually predicts.

Explanations are part of model inspection. They do not by themselves establish fairness, safety or suitability for deployment.

The code converts numerical features to floating-point values with `.astype(float)`. Partial-dependence calculations replace them with grid values that can fall between the observed integers, including fractional months.

In [ ]:
resale = pd.read_csv('../data/resale_price_dataset.csv')
features = ['Retail price', 'Actual Lease Duration (months)', 'Battery capacity (%)']
X = resale[features].astype(float)
y = resale['Observed resale price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = GradientBoostingRegressor(
    n_estimators=100, max_depth=2, learning_rate=0.05, random_state=42
)
model.fit(X_train, y_train)
print(f'Test MAE: {mean_absolute_error(y_test, model.predict(X_test)):.2f} USD')
X_test.head()

The fitted model stays fixed throughout the explanations. The first 200 held-out rows form a manageable reference set for plots and SHAP calculations. Their selection does not depend on prediction errors or labels. Global findings below describe this subset, not necessarily every possible laptop.

Retaining the row indices keeps observations, predictions and explanations aligned.

In [ ]:
X_explain = X_test.iloc[:200].copy()
y_explain = y_test.loc[X_explain.index]
print(f'Explanation reference rows: {len(X_explain)}')
X_explain.corr().round(2)

The correlation table provides context for the synthetic feature combinations used below. Low pairwise correlations do not guarantee that all combinations are realistic; strong dependence makes independent replacements especially problematic.

## Global explanation methods

### A simple surrogate: explain the model, not the labels

The next cell fits a small regression tree to **ensemble predictions** on the training features. This tree is a *surrogate*: it summarizes an opaque predictor using a short set of rules. Its training target is not the observed resale price.

Its fidelity is measured on held-out rows by comparing surrogate predictions with ensemble predictions. Fidelity asks “does the surrogate mimic this model?”, whereas ordinary predictive MAE asks “does the model predict actual prices?”

In [ ]:
surrogate = DecisionTreeRegressor(max_depth=3, min_samples_leaf=50, random_state=42)
surrogate.fit(X_train, model.predict(X_train))
fidelity_mae = mean_absolute_error(model.predict(X_explain), surrogate.predict(X_explain))
print(f'Surrogate fidelity MAE: {fidelity_mae:.2f} USD')
print(export_text(surrogate, feature_names=features, decimals=1))

The rules are readable, but their fidelity error quantifies what they leave out. The surrogate is therefore an approximation, not an exact account of every ensemble prediction.

### Permutation importance

![XAI lecture slide 12: corrupting one feature tests how strongly a fixed model relies on its information.](../resources/6_permutation_importance.png)

`permutation_importance` shuffles one column at a time and reevaluates the model. The example specifies `scoring='neg_mean_absolute_error'`: **scikit-learn** expects larger scores to be better, so it negates MAE. The reported score decrease therefore equals the **increase in MAE**, in USD, after shuffling.

Five repetitions of each shuffle show sensitivity to the random permutations. A negative result means the shuffled predictions happened to improve this subset's MAE, not that the feature has a negative causal effect.

In [ ]:
importance = permutation_importance(
    model, X_explain, y_explain, scoring='neg_mean_absolute_error',
    n_repeats=5, random_state=42
)
importance_table = pd.DataFrame({
    'feature': features,
    'mean_mae_increase_usd': importance.importances_mean,
    'shuffle_std_usd': importance.importances_std
}).sort_values('mean_mae_increase_usd')
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importance_table['feature'], importance_table['mean_mae_increase_usd'],
        xerr=importance_table['shuffle_std_usd'], capsize=4)
ax.axvline(0, color='gray', linewidth=1)
ax.set(xlabel='Increase in MAE after shuffling (USD)', title='Permutation importance on held-out rows')
plt.tight_layout()
plt.show()
importance_table.round(2)

Longer bars indicate stronger reliance under this metric and reference sample. The error bars summarize repeated shuffles on these same rows; they are not confidence intervals over future populations. Correlated features may share or substitute for predictive information, so rankings are not universal properties of the features.

### Partial dependence and individual curves

The next cell calculates three average forecasts after replacing lease duration with 12, 36 and 60 months, making the PDP calculation explicit. All other columns stay at their reference values. These are hypothetical modified rows, not observed outcomes.

In [ ]:
duration_feature = 'Actual Lease Duration (months)'
pdp_rows = []
for duration in [12, 36, 60]:
    modified = X_explain.copy()
    modified[duration_feature] = duration
    pdp_rows.append({'duration_months': duration, 'average_forecast_usd': model.predict(modified).mean()})
pd.DataFrame(pdp_rows).round(2)

`PartialDependenceDisplay` extends this calculation over a grid. `kind='both'` draws the average PDP and individual ICE curves; `subsample=40` limits the displayed individual curves. The PDP still averages over the supplied reference set. `method='brute'` computes predictions on modified rows directly, matching the calculation above.

The returned display stores the actual plotting axes in `axes_`. The code sets the labels on its first axis so that the feature and price units appear on the plot.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
pdp_display = PartialDependenceDisplay.from_estimator(
    model, X_explain, features=[duration_feature], kind='both',
    subsample=40, random_state=42, method='brute', grid_resolution=30, ax=ax
)
pdp_display.axes_[0, 0].set(xlabel='Actual lease duration (months)', ylabel='Predicted resale price (USD)',
       title='Average response and individual prediction curves')
plt.tight_layout()
plt.show()

Read the average direction and compare the individual curves. Parallel curves suggest similar responses over the plotted range; differences in their shapes suggest interactions in the fitted model. The small marks on the horizontal axis indicate feature deciles.

![XAI lecture slide 22: dependence between features and heterogeneous responses limit what an average PDP can show.](../resources/6_pdp_limits.png)

This describes a response of the **fitted prediction function**, not a causal depreciation estimate. Replacing one feature can create combinations absent from training, and an average can hide individual variation. A causal conclusion needs assumptions and evidence beyond this plot.

## Local and example-based explanations

### SHAP: allocate the difference from a baseline

![XAI lecture slide 27: feature contributions move a baseline prediction toward an individual forecast.](../resources/6_shap_example.png)

The slide illustrates a credit-class probability; our example explains a **resale-price forecast in USD**. Positive contributions push the price above the reference level, and negative contributions push it below.

`TreeExplainer` exploits the tree structure. Our example supplies 100 sampled training rows as the background and explicitly sets `feature_perturbation='interventional'`. Missing-feature contributions are defined using replacements from this background. The term *interventional* names a computational convention here; it does not establish a causal interpretation.

For this regression model, `model_output='raw'` is the price prediction itself. With a binary classifier, output units can instead be probabilities or log-odds depending on the model and explainer settings, so the output scale depends on that configuration.

In [ ]:
background = X_train.sample(n=100, random_state=42)
explainer = shap.TreeExplainer(
    model, data=background, feature_perturbation='interventional', model_output='raw'
)
shap_values = explainer(X_explain)
print('SHAP array shape (rows, features):', shap_values.values.shape)
print(f'Baseline forecast: {shap_values.base_values[0]:.2f} USD')
print(f'Mean model forecast on background: {model.predict(background).mean():.2f} USD')

The `Explanation` object keeps feature values, contribution values and baselines together. In this regression problem, `.values` has one column per feature; no class dimension needs selecting.

The next cell checks the additive accounting before the contributions are plotted. Summing across `axis=1` adds feature contributions for each explained row.

In [ ]:
reconstructed = shap_values.base_values + shap_values.values.sum(axis=1)
model_forecast = model.predict(X_explain)
print(f'Largest reconstruction difference: {np.abs(reconstructed - model_forecast).max():.6f} USD')
local_row = 0
local_table = pd.DataFrame({
    'feature_value': X_explain.iloc[local_row],
    'contribution_usd': shap_values.values[local_row]
})
print(f'Explained row index: {X_explain.index[local_row]}')
print(f'Forecast: {model_forecast[local_row]:.2f} USD; actual price: {y_explain.iloc[local_row]:.2f} USD')
local_table.round(2)

In [ ]:
shap.plots.waterfall(shap_values[local_row], max_display=3, show=False)
plt.gcf().set_size_inches(11, 4)
# Keep the USD scale consistent on the main and annotation axes.
for axis in plt.gcf().axes:
    axis.set_xlim(0, max(shap_values.base_values[local_row], model_forecast[local_row]) * 1.15)
plt.title('One laptop: contributions to predicted resale price (USD)')
plt.tight_layout()
plt.show()

The waterfall plot starts at the baseline and adds the feature contributions to arrive at the forecast. This is not an allocation of prediction error: the actual sale price is a separate observed quantity. The baseline is an average **model forecast** over the chosen background, not necessarily the mean observed sale price.

### Aggregate local contributions

A beeswarm plot places one contribution per feature and explained row. Horizontal position is the SHAP contribution in USD; color indicates that feature's value. The ranking uses mean absolute SHAP magnitude. It answers a different question from permutation importance, which measures lost predictive performance after corruption.

In [ ]:
shap.plots.beeswarm(shap_values, max_display=3, show=False, plot_size=(8, 4.5))
plt.xlabel('SHAP contribution to predicted price (USD)')
plt.title('Local contributions across the explanation reference set')
plt.tight_layout()
plt.show()

Inspect whether high feature values tend to have positive or negative contributions. The ranking need not match permutation importance: one summarizes output differences relative to a background, while the other summarizes changes in error after shuffling.

### Explain through modified examples

The final example constructs two hypothetical return profiles with the same retail price and battery capacity but different lease durations. This illustrates a *what-if* comparison. It is not a claim that a borrower can change elapsed time, nor a search for the smallest change that reaches a target output.

A strict *counterfactual explanation* would specify a desired model output, a distance measure and feasibility constraints, then search for a change satisfying them. The lecture also distinguishes adversarial modifications whose purpose is to induce errors.

In [ ]:
profiles = pd.DataFrame({
    'Retail price': [2200, 2200],
    'Actual Lease Duration (months)': [24, 36],
    'Battery capacity (%)': [90, 90]
}, index=['Return after 24 months', 'Return after 36 months'])
profiles['predicted_price_usd'] = model.predict(profiles[features])
profiles.round(2)

These forecasts describe model responses to the specified inputs. Neither SHAP nor this comparison proves what the actual sale-price difference would be under a real intervention.

## Summary

- Different audiences and questions call for different explanations: global reliance, average response or one local prediction.
- A surrogate's fidelity concerns agreement with a model, not with true outcomes.
- Permutation importance depends on the metric and reference rows; it is not a retraining experiment.
- PDP averages modified-row predictions; ICE reveals individual responses. Neither automatically identifies causality.
- SHAP contributions add to the prediction relative to a specified background and output scale.
- Explanations describe a fixed model and do not establish fairness, business value or feasibility by themselves.

Here's a quick reference for the new Python tools introduced in this notebook:

| Python pattern | What it helps you do |
|---|---|
| `.corr()` | Inspect dependence between explanation features |
| `permutation_importance` | Measure the increase in price error after shuffling a feature |
| `PartialDependenceDisplay.from_estimator(..., kind='both')` | Display average and individual model responses |
| `.sample(n=100, random_state=42)` | Select a reproducible explanation background |
| `shap.TreeExplainer` | Calculate additive tree-model explanations |
| `Explanation.base_values`, `Explanation.values` | Retrieve the baseline and feature contributions for reconstruction |
| `shap.plots.waterfall`, `shap.plots.beeswarm` | Inspect one explanation and aggregate local explanations |
| `ax.barh(..., xerr=...)` | Plot importance and shuffle variation in explicit units |

## Tasks

### Task 1: Explain another laptop

Choose a different positional row from `X_explain`. Report its baseline, each contribution and the reconstructed forecast, then make a waterfall plot. Explain the largest contribution in words without claiming it caused the actual sale price.

In [ ]:
# YOUR CODE: explain and reconstruct another forecast.

### Task 2: Make the PDP calculation explicit

For battery capacities 70, 85 and 95 percent, replace the battery column in a copy of `X_explain` and calculate the average forecast. Compare those values with a battery PDP. Discuss whether all modified profiles are well supported by the original data.

In [ ]:
# YOUR CODE: manually calculate three battery partial-dependence values.

### Task 3: Check the surrogate locally

Compare the surrogate and ensemble forecast for three rows in `X_explain`. Show their absolute differences and explain why a low average fidelity error would not guarantee that the surrogate is accurate for every individual row.

*Hint:* use `surrogate.predict` and `model.predict` on the same rows.

In [ ]:
# YOUR CODE: inspect individual surrogate discrepancies.